# Part 4 — Extraction & Entity Resolution

This notebook demonstrates the correct functioning of the two-stage Agentic Entity Resolution pipeline (EP-03, EP-04):

| File | Responsibility |
|------|----------------|
| `src/extraction/triplet_extractor.py` | SLM JSON-mode extraction of (subject, predicate, object) triplets |
| `src/resolution/blocking.py` | Stage 1 — K-NN vector blocking to group near-duplicate entities |
| `src/resolution/llm_judge.py` | Stage 2 — LLM judge decides merge/separate for each cluster |
| `src/resolution/entity_resolver.py` | Orchestrator: blocking → judge → canonical `Entity` objects |

**Architecture overview:**
```
Chunk text
    │
    ▼
extract_triplets()  [SLM, T=0.0, JSON mode]
    │  → list[Triplet]   (subject, predicate, object, provenance, confidence)
    │
    ▼
resolve_entities()
    ├── Stage 1: block_entities()   [BGE-M3 embeddings + cosine similarity + Union-Find]
    │   → list[EntityCluster]
    └── Stage 2: judge_cluster()    [LLM reasoning, T=0.0]
        → list[Entity]   (canonical, deduplicated, with synonyms)
```
> **Note**: LLM and embedding calls require live credentials. Sections that perform actual LLM/embedding
> inference use mocks or pre-computed examples so the notebook runs fully offline.

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. `extract_triplets()` — SLM-based triplet extraction

`extract_triplets(chunk, llm)` calls the SLM with `EXTRACTION_SYSTEM` / `EXTRACTION_USER` in JSON mode.
All failures (LLM errors, JSON parse errors, Pydantic validation errors) are caught and logged — the function
returns `[]` rather than raising. `extract_all_triplets(chunks, llm)` batches across all chunks.

In [2]:
from unittest.mock import MagicMock
from langchain_core.messages import AIMessage
from src.models.schemas import Chunk, Triplet
from src.extraction.triplet_extractor import extract_triplets, extract_all_triplets

# Build a sample chunk from the business glossary
chunk = Chunk(
    text="A Customer places one or more SalesOrders. A SalesOrder contains one or more OrderLineItems.",
    chunk_index=0,
    metadata={"source": "business_glossary.pdf", "page": "1", "token_count": "20"},
)

# Mock LLM returning a valid JSON triplet response
mock_llm = MagicMock()
mock_llm.invoke.return_value = AIMessage(content='''
{
  "triplets": [
    {
      "subject": "Customer",
      "predicate": "places",
      "object": "SalesOrder",
      "provenance_text": "A Customer places one or more SalesOrders.",
      "confidence": 1.0
    },
    {
      "subject": "SalesOrder",
      "predicate": "contains",
      "object": "OrderLineItem",
      "provenance_text": "A SalesOrder contains one or more OrderLineItems.",
      "confidence": 1.0
    }
  ]
}
''')

triplets = extract_triplets(chunk, mock_llm)

print(f"Triplets extracted from chunk {chunk.chunk_index}: {len(triplets)}")
for t in triplets:
    print(f"  ({t.subject}, {t.predicate}, {t.object}) [conf={t.confidence}, chunk={t.source_chunk_index}]")

{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 0 \u2192 2 triplets extracted (source=business_glossary.pdf, page=1)"}


Triplets extracted from chunk 0: 2
  (Customer, places, SalesOrder) [conf=1.0, chunk=0]
  (SalesOrder, contains, OrderLineItem) [conf=1.0, chunk=0]


In [3]:
# Verify graceful failure handling

# 1. LLM returns invalid JSON → returns []
mock_llm_bad_json = MagicMock()
mock_llm_bad_json.invoke.return_value = AIMessage(content="This is not JSON at all")
result = extract_triplets(chunk, mock_llm_bad_json)
assert result == [], "Should return [] on JSON parse error"
print("✅ Invalid JSON → gracefully returns []")

# 2. LLM returns JSON that fails Pydantic validation → returns []
mock_llm_bad_schema = MagicMock()
mock_llm_bad_schema.invoke.return_value = AIMessage(content='{"triplets": [{"wrong_field": 42}]}')
result = extract_triplets(chunk, mock_llm_bad_schema)
assert result == [], "Should return [] on Pydantic ValidationError"
print("✅ Pydantic ValidationError → gracefully returns []")

# 3. LLM call raises exception → returns []
mock_llm_error = MagicMock()
mock_llm_error.invoke.side_effect = RuntimeError("Connection error")
result = extract_triplets(chunk, mock_llm_error)
assert result == [], "Should return [] on LLM exception"
print("✅ LLM exception → gracefully returns []")

# 4. source_chunk_index is correctly attached to each triplet
for t in triplets:
    assert t.source_chunk_index == chunk.chunk_index
print(f"✅ source_chunk_index={triplets[0].source_chunk_index} correctly attached")

{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "WARNING", "message": "Non-JSON response for chunk 0: Expecting value: line 1 column 1 (char 0). Raw response: 'This is not JSON at all'"}
{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "WARNING", "message": "Pydantic validation failed for chunk 0: 5 validation errors for TripletExtractionResponse\ntriplets.0.subject\n  Field required [type=missing, input_value={'wrong_field': 42}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\ntriplets.0.predicate\n  Field required [type=missing, input_value={'wrong_field': 42}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\ntriplets.0.object\n  Field required [type=missing, input_value={'wrong_field': 42}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\ntriplets.0.provenance_text\n  Fie

✅ Invalid JSON → gracefully returns []
✅ Pydantic ValidationError → gracefully returns []
✅ LLM exception → gracefully returns []
✅ source_chunk_index=0 correctly attached


In [4]:
# extract_all_triplets: batch extraction across all chunks
chunks = [
    Chunk(
        text="A Customer places one or more SalesOrders.",
        chunk_index=i,
        metadata={"source": "bg.pdf", "page": str(i+1), "token_count": "10"},
    )
    for i in range(3)
]

# Each chunk returns 1 triplet
single_triplet_json = '{"triplets": [{"subject": "Customer", "predicate": "places", "object": "SalesOrder", "provenance_text": "A Customer places one or more SalesOrders.", "confidence": 1.0}]}'
mock_batch_llm = MagicMock()
mock_batch_llm.invoke.return_value = AIMessage(content=single_triplet_json)

all_triplets = extract_all_triplets(chunks, mock_batch_llm)

print(f"extract_all_triplets: {len(all_triplets)} triplets from {len(chunks)} chunks")
print(f"  LLM invoked: {mock_batch_llm.invoke.call_count} time(s) (once per chunk)")
print(f"  source_chunk_indices: {[t.source_chunk_index for t in all_triplets]}")
assert len(all_triplets) == 3
print("✅ extract_all_triplets batches correctly")

{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 0 \u2192 1 triplets extracted (source=bg.pdf, page=1)"}
{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 1 \u2192 1 triplets extracted (source=bg.pdf, page=2)"}
{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 2 \u2192 1 triplets extracted (source=bg.pdf, page=3)"}
{"ts": "2026-03-12T12:07:33", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Extracted 3 total triplets from 3 chunks."}


extract_all_triplets: 3 triplets from 3 chunks
  LLM invoked: 3 time(s) (once per chunk)
  source_chunk_indices: [0, 1, 2]
✅ extract_all_triplets batches correctly


## 2. Stage 1: `block_entities()` — Vector-based K-NN blocking

`block_entities(entities, embeddings)` embeds all unique entity strings, computes the full N×N cosine
similarity matrix, and uses Union-Find to group near-duplicate pairs into `EntityCluster` objects.

**Algorithm:**
1. Embed all entity strings in one batched call (BGE-M3 recommended)
2. Compute N×N cosine similarity matrix (`sklearn`)
3. For each entity, collect pairs with similarity ≥ `settings.er_similarity_threshold`
4. Apply Union-Find to merge overlapping pairs into clusters
5. Return only clusters with ≥ 2 variants (singletons are not ambiguous)

In [5]:
import numpy as np
from unittest.mock import MagicMock
from src.resolution.blocking import extract_unique_entities, block_entities
from src.models.schemas import Triplet, EntityCluster

# Build a set of triplets with intentional near-duplicate entity names
sample_triplets = [
    Triplet(subject="Customer",  predicate="places",    object="SalesOrder",  provenance_text="A Customer places a SalesOrder.",       confidence=1.0),
    Triplet(subject="customer",  predicate="places",    object="Order",       provenance_text="A customer places an Order.",           confidence=0.9),
    Triplet(subject="Client",    predicate="has",       object="Address",     provenance_text="A Client has a CustomerAddress.",        confidence=0.9),
    Triplet(subject="Product",   predicate="belongs to",object="Category",    provenance_text="A Product belongs to a Category.",       confidence=1.0),
    Triplet(subject="Inventory", predicate="tracks",    object="Product",     provenance_text="Inventory tracks available Products.",   confidence=0.8),
]

unique_entities = extract_unique_entities(sample_triplets)
print(f"Unique entities extracted: {len(unique_entities)}")
for e in unique_entities:
    print(f"  · '{e}'")

/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Extracted 9 unique entities from 5 triplets."}


Unique entities extracted: 9
  · 'Address'
  · 'Category'
  · 'Client'
  · 'Customer'
  · 'Inventory'
  · 'Order'
  · 'Product'
  · 'SalesOrder'
  · 'customer'


In [6]:
# Mock embeddings that return high similarity for semantic duplicates
# Strategy: assign similar vectors to 'Customer', 'customer', 'Client' and 'SalesOrder'/'Order'

entity_list = sorted(unique_entities)
print(f"Entities for blocking (sorted): {entity_list}")

# Build a mock embedding space in 4 dimensions:
# Customer-variants cluster: Customer, customer, Client (all near [1,0,0,0])
# Order-variants cluster: SalesOrder, Order (near [0,1,0,0])
# Others: isolated
customer_like = {"Customer", "customer", "Client"}
order_like = {"SalesOrder", "Order"}

def mock_embed(texts):
    vecs = []
    for t in texts:
        if t in customer_like:
            vecs.append([0.98, 0.10, 0.05, 0.00])
        elif t in order_like:
            vecs.append([0.05, 0.97, 0.10, 0.00])
        elif t == "Address":
            vecs.append([0.00, 0.00, 0.95, 0.10])
        else:
            vecs.append([0.00, 0.00, 0.00, 0.99])
    # L2-normalise so cosine similarity works correctly
    vecs = np.array(vecs, dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return (vecs / norms).tolist()

mock_embeddings = MagicMock()
mock_embeddings.embed_documents.side_effect = mock_embed

# Run blocking with a lower threshold to ensure our mock clusters form
clusters = block_entities(entity_list, mock_embeddings, threshold=0.85, top_k=5)

print(f"\nClusters found: {len(clusters)}")
for c in clusters:
    print(f"\n  Cluster [canonical='{c.canonical_candidate}', avg_sim={c.avg_similarity}]:")
    for v in c.variants:
        print(f"    · '{v}'")

assert isinstance(clusters[0], EntityCluster)
print("\n✅ block_entities produced EntityCluster objects")

{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Embedding 9 entities for blocking..."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Blocking complete: 3 clusters from 9 entities (threshold=0.85)."}


Entities for blocking (sorted): ['Address', 'Category', 'Client', 'Customer', 'Inventory', 'Order', 'Product', 'SalesOrder', 'customer']

Clusters found: 3

  Cluster [canonical='Inventory', avg_sim=1.0]:
    · 'Category'
    · 'Inventory'
    · 'Product'

  Cluster [canonical='Customer', avg_sim=1.0]:
    · 'Client'
    · 'Customer'
    · 'customer'

  Cluster [canonical='SalesOrder', avg_sim=1.0]:
    · 'Order'
    · 'SalesOrder'

✅ block_entities produced EntityCluster objects


## 3. Stage 2: `judge_cluster()` — LLM canonicalisation judge

`judge_cluster(cluster, provenance_map, llm)` calls the LLM with `ER_JUDGE_SYSTEM` / `ER_JUDGE_USER`.
The LLM decides whether all variants in the cluster refer to the **same** real-world concept (`merge=True`)
or should remain separate (`merge=False`). On any failure, a conservative *no-merge* decision is returned.

In [7]:
from src.resolution.llm_judge import build_provenance_map, judge_cluster, cluster_to_entity
from src.models.schemas import EntityCluster, CanonicalEntityDecision, Entity

# Build provenance map from the sample triplets
provenance_map = build_provenance_map(sample_triplets)
print("Provenance map (entity → provenance texts):")
for entity, texts in list(provenance_map.items())[:4]:
    print(f"  '{entity}': {texts[0][:60]}…")

Provenance map (entity → provenance texts):
  'Customer': A Customer places a SalesOrder.…
  'SalesOrder': A Customer places a SalesOrder.…
  'customer': A customer places an Order.…
  'Order': A customer places an Order.…


In [8]:
# Simulate judge for the Customer cluster (merge=True case)
customer_cluster = EntityCluster(
    canonical_candidate="Customer",
    variants=["Customer", "customer", "Client"],
    avg_similarity=0.97,
)

# Mock LLM returning a merge=True decision
merge_llm = MagicMock()
merge_llm.invoke.return_value = AIMessage(content='''
{"merge": true, "canonical_name": "Customer", "reasoning": "All variants refer to the same platform customer entity."}
''')

decision = judge_cluster(customer_cluster, provenance_map, merge_llm)

print("LLM Judge decision (merge=True):")
print(f"  merge:          {decision.merge}")
print(f"  canonical_name: '{decision.canonical_name}'")
print(f"  reasoning:      '{decision.reasoning[:70]}'")
assert decision.merge is True
assert decision.canonical_name == "Customer"

{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "INFO", "message": "ER judge: cluster 'Customer' \u2192 merge=True, canonical='Customer'"}


LLM Judge decision (merge=True):
  merge:          True
  canonical_name: 'Customer'
  reasoning:      'All variants refer to the same platform customer entity.'


In [9]:
# Simulate judge for a no-merge case (distinct concepts)
mixed_cluster = EntityCluster(
    canonical_candidate="Category",
    variants=["Category", "Inventory"],
    avg_similarity=0.86,
)

nomerge_llm = MagicMock()
nomerge_llm.invoke.return_value = AIMessage(content='''
{"merge": false, "canonical_name": "Category", "reasoning": "Category and Inventory refer to different concepts."}
''')

decision_nomerge = judge_cluster(mixed_cluster, provenance_map, nomerge_llm)
print("LLM Judge decision (merge=False):")
print(f"  merge:          {decision_nomerge.merge}")
print(f"  canonical_name: '{decision_nomerge.canonical_name}'")
assert decision_nomerge.merge is False

# Conservative no-merge fallback on LLM failure
error_llm = MagicMock()
error_llm.invoke.side_effect = RuntimeError("LLM unreachable")
fallback = judge_cluster(customer_cluster, provenance_map, error_llm)
print(f"\nFallback on LLM error: merge={fallback.merge} (conservative no-merge)")
assert fallback.merge is False
print("✅ Conservative no-merge fallback on any failure")

{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "INFO", "message": "ER judge: cluster 'Category' \u2192 merge=False, canonical='Category'"}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "WARNING", "message": "LLM judge call failed for cluster Customer: LLM unreachable \u2014 defaulting to no-merge."}


LLM Judge decision (merge=False):
  merge:          False
  canonical_name: 'Category'

Fallback on LLM error: merge=False (conservative no-merge)
✅ Conservative no-merge fallback on any failure


In [10]:
# cluster_to_entity: convert cluster + decision into a canonical Entity
entity = cluster_to_entity(customer_cluster, decision, provenance_map)

print("Canonical Entity from cluster_to_entity():")
print(f"  name:           '{entity.name}'")
print(f"  synonyms:       {entity.synonyms}")
print(f"  provenance:     '{entity.provenance_text[:80]}...'" if len(entity.provenance_text) > 80 else f"  provenance: '{entity.provenance_text}'")
assert entity.name == "Customer"
assert "customer" in entity.synonyms or "Client" in entity.synonyms
print("✅ cluster_to_entity correctly builds the Entity")

Canonical Entity from cluster_to_entity():
  name:           'Customer'
  synonyms:       ['customer', 'Client']
  provenance:     'A Customer places a SalesOrder. | A customer places an Order. | A Client has a C...'
✅ cluster_to_entity correctly builds the Entity


## 4. Full pipeline: `resolve_entities()`

`resolve_entities(triplets, embeddings, llm, source_doc)` orchestrates the two stages:
1. Extracts unique entities from all triplets
2. Runs Stage 1 blocking → `list[EntityCluster]`
3. Runs Stage 2 judge for each cluster → merge/keep decision
4. Promotes singletons (not in any cluster) directly to `Entity` objects

The result is a flat `list[Entity]` ready to embed and store in `BuilderState`.

In [11]:
from src.resolution.entity_resolver import resolve_entities

# LLM mock: always merges (merge=True, picks the canonical_candidate)
def make_merge_response(cluster_name):
    import json
    payload = {"merge": True, "canonical_name": cluster_name, "reasoning": "Same concept."}
    return AIMessage(content=json.dumps(payload))

judge_llm = MagicMock()
judge_llm.invoke.side_effect = lambda msgs, **kw: make_merge_response(
    # parse canonical from cluster in message content — simplified: always return 'Customer'
    "Customer"
)

entities = resolve_entities(
    triplets=sample_triplets,
    embeddings=mock_embeddings,
    llm=judge_llm,
    source_doc="business_glossary.pdf",
)

print(f"resolve_entities() → {len(entities)} canonical entities:")
for e in entities:
    synonyms_str = f" (synonyms: {e.synonyms})" if e.synonyms else ""
    print(f"  · '{e.name}'{synonyms_str}")
    print(f"    source_doc='{e.source_doc}'")

{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Extracted 9 unique entities from 5 triplets."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Embedding 9 entities for blocking..."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Blocking complete: 3 clusters from 9 entities (threshold=0.85)."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "INFO", "message": "ER judge: cluster 'Inventory' \u2192 merge=True, canonical='Customer'"}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "INFO", "message": "ER judge: cluster 'Customer' \u2192 merge=True, canonical='Customer'"}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "INFO", "message": "ER judge: cluster 'SalesOrder' \u2192 merge=True, canonical='Customer'"}
{"ts": "2026-03-12T12:07:34", "logger": "src.resoluti

resolve_entities() → 4 canonical entities:
  · 'Customer' (synonyms: ['Category', 'Inventory', 'Product'])
    source_doc='business_glossary.pdf'
  · 'Customer' (synonyms: ['Client', 'customer'])
    source_doc='business_glossary.pdf'
  · 'Customer' (synonyms: ['Order', 'SalesOrder'])
    source_doc='business_glossary.pdf'
  · 'Address'
    source_doc='business_glossary.pdf'


In [12]:
# Validate Entity objects
from src.models.schemas import Entity

for e in entities:
    assert isinstance(e, Entity)
    assert e.source_doc == "business_glossary.pdf"
    assert isinstance(e.synonyms, list)

print(f"✅ All {len(entities)} entities:")
print(f"   · are Entity instances")
print(f"   · have source_doc='business_glossary.pdf'")
print(f"   · have synonyms list")

# Edge case: empty triplet list
empty_result = resolve_entities([], mock_embeddings, judge_llm)
assert empty_result == []
print("✅ Empty triplet list → returns []")

{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.entity_resolver", "level": "WARNING", "message": "resolve_entities called with empty triplet list \u2014 returning empty."}


✅ All 4 entities:
   · are Entity instances
   · have source_doc='business_glossary.pdf'
   · have synonyms list
✅ Empty triplet list → returns []


## 5. End-to-end pipeline simulation

Complete flow: raw text `→` chunks `→` triplets `→` entities, as it will run inside the LangGraph
`BuilderState` during the Knowledge Graph construction.

In [13]:
from src.models.schemas import Chunk, Triplet, Entity
from src.extraction.triplet_extractor import extract_all_triplets
from src.resolution.entity_resolver import resolve_entities

# Simulated chunks from the business glossary
raw_chunks = [
    Chunk(text="A Customer places one or more SalesOrders. Each SalesOrder has a unique Order ID.",
          chunk_index=0, metadata={"source": "bg.pdf", "page": "1", "token_count": "18"}),
    Chunk(text="A SalesOrder contains one or more OrderLineItems referencing Products.",
          chunk_index=1, metadata={"source": "bg.pdf", "page": "2", "token_count": "12"}),
    Chunk(text="Each Product belongs to exactly one Category. Inventory tracks available stock per Product.",
          chunk_index=2, metadata={"source": "bg.pdf", "page": "3", "token_count": "15"}),
]

# Mock extraction LLM — returns realistic triplets for each chunk
extraction_responses = [
    '{"triplets":[{"subject":"Customer","predicate":"places","object":"SalesOrder","provenance_text":"A Customer places one or more SalesOrders.","confidence":1.0}]}',
    '{"triplets":[{"subject":"SalesOrder","predicate":"contains","object":"OrderLineItem","provenance_text":"A SalesOrder contains one or more OrderLineItems.","confidence":1.0}]}',
    '{"triplets":[{"subject":"Product","predicate":"belongs to","object":"Category","provenance_text":"Each Product belongs to exactly one Category.","confidence":1.0},{"subject":"Inventory","predicate":"tracks","object":"Product","provenance_text":"Inventory tracks available stock per Product.","confidence":0.9}]}',
]
extraction_llm = MagicMock()
extraction_llm.invoke.side_effect = [AIMessage(content=r) for r in extraction_responses]

# Stage 1: Extract triplets
all_triplets = extract_all_triplets(raw_chunks, extraction_llm)
print(f"Step 1 — Triplets extracted: {len(all_triplets)}")
for t in all_triplets:
    print(f"  chunk={t.source_chunk_index}: ({t.subject}, {t.predicate}, {t.object}) [conf={t.confidence}]")

{"ts": "2026-03-12T12:07:34", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 0 \u2192 1 triplets extracted (source=bg.pdf, page=1)"}
{"ts": "2026-03-12T12:07:34", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 1 \u2192 1 triplets extracted (source=bg.pdf, page=2)"}
{"ts": "2026-03-12T12:07:34", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Chunk 2 \u2192 2 triplets extracted (source=bg.pdf, page=3)"}
{"ts": "2026-03-12T12:07:34", "logger": "src.extraction.triplet_extractor", "level": "INFO", "message": "Extracted 4 total triplets from 3 chunks."}


Step 1 — Triplets extracted: 4
  chunk=0: (Customer, places, SalesOrder) [conf=1.0]
  chunk=1: (SalesOrder, contains, OrderLineItem) [conf=1.0]
  chunk=2: (Product, belongs to, Category) [conf=1.0]
  chunk=2: (Inventory, tracks, Product) [conf=0.9]


In [14]:
import json

# Stage 2: Entity Resolution
# Mock judge LLM: returns the first variant as canonical (merge=True)
def smart_judge(msgs, **kw):
    # Parse variants from the HumanMessage content
    content = msgs[-1].content
    import re
    m = re.search(r'\[(.+?)\]', content)
    if m:
        variants = json.loads('[' + m.group(1) + ']')
        canonical = max(variants, key=len)  # pick the longest as canonical
        return AIMessage(content=json.dumps({"merge": True, "canonical_name": canonical, "reasoning": "Same concept."}))
    return AIMessage(content=json.dumps({"merge": False, "canonical_name": "Unknown", "reasoning": "Could not parse."}))

resolution_llm = MagicMock()
resolution_llm.invoke.side_effect = smart_judge

canonical_entities = resolve_entities(
    triplets=all_triplets,
    embeddings=mock_embeddings,
    llm=resolution_llm,
    source_doc="bg.pdf",
)

print(f"\nStep 2 — Canonical entities: {len(canonical_entities)}")
for e in canonical_entities:
    synonyms_str = f" [synonyms: {e.synonyms}]" if e.synonyms else ""
    print(f"  · '{e.name}'{synonyms_str}")

print(f"\n  LLM judge calls: {resolution_llm.invoke.call_count}")
print(f"  Reduction: {len(all_triplets)*2} raw entity strings → {len(canonical_entities)} canonical entities")

{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Extracted 6 unique entities from 4 triplets."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Embedding 6 entities for blocking..."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.blocking", "level": "INFO", "message": "Blocking complete: 1 clusters from 6 entities (threshold=0.85)."}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.llm_judge", "level": "INFO", "message": "ER judge: cluster 'OrderLineItem' \u2192 merge=True, canonical='OrderLineItem'"}
{"ts": "2026-03-12T12:07:34", "logger": "src.resolution.entity_resolver", "level": "INFO", "message": "Entity resolution complete: 1 clusters resolved, 2 singletons promoted \u2192 3 total entities."}



Step 2 — Canonical entities: 3
  · 'OrderLineItem' [synonyms: ['Category', 'Inventory', 'Product']]
  · 'Customer'
  · 'SalesOrder'

  LLM judge calls: 1
  Reduction: 8 raw entity strings → 3 canonical entities


## Summary — Part 4 Architecture

```
┌────────────────────────────────────────────────────────────────────────┐
│              EP-03 + EP-04 — Extraction & Entity Resolution            │
├───────────────────────┬────────────────────────────────────────────────┤
│  triplet_extractor.py │       resolution/                              │
│                       │  blocking.py │ llm_judge.py │ entity_resolver  │
│  extract_triplets()   │              │              │                  │
│  · SLM JSON mode      │  Stage 1:    │  Stage 2:    │ Orchestrator:    │
│  · T=0.0              │  embed all   │  LLM judge   │ blocking         │
│  · graceful []        │  entities    │  merge/keep  │ → judge          │
│    on any failure     │  cosine sim  │  conservative│ → singletons     │
│                       │  Union-Find  │  no-merge    │ → list[Entity]   │
│  extract_all_         │  → clusters  │  fallback    │                  │
│    triplets()         │              │              │                  │
└───────────────────────┴──────────────┴──────────────┴──────────────────┘
```

**Key design decisions:**
- **Graceful degradation**: every LLM call is wrapped in try/except; failures return safe defaults
- **Conservative no-merge**: LLM judge failures always choose *not* to merge, preserving information
- **Singleton promotion**: entities not in any cluster skip the LLM judge entirely (no wasted calls)
- **Provenance-grounded**: the LLM judge receives verbatim context sentences, not just entity names
- **Composable**: `extract_triplets` and `resolve_entities` are pure functions usable independently of LangGraph